> [!WARNING]
> **UNRUN NOTEBOOK; NO RESULTS CLAIMED.**
> This notebook is an unexecuted template designed for cloud GPU environments (Kaggle T4 or Google Colab). It fine-tunes `microsoft/deberta-v3-small` on `data/splits/train.jsonl.gz` (fields: `text`, `label`) and exports a dynamically quantized INT8 ONNX model for low-end CPU deployment.

# 🚀 Veritas AI — Notebook 3: GPU Fine-Tuning & ONNX INT8 Export

**Architecture Target**: Cloud GPU (Kaggle / Colab) -> Export to CPU Edge Deployment


In [ ]:
# 1. Install Dependencies
!pip install -q torch transformers datasets accelerate onnx onnxruntime scikit-learn

import os
import gzip
import json
import numpy as np
import torch
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
)
from datasets import Dataset
from sklearn.metrics import accuracy_score, f1_score
import onnx
from onnxruntime.quantization import quantize_dynamic, QuantType

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Active device: {device}')


## 2. Load Dataset Splits

Loads `data/splits/train.jsonl.gz` and `data/splits/dev.jsonl.gz`. Each record contains `text` (string) and `label` (integer or class string).

In [ ]:
def load_jsonl_gz(filepath):
    records = []
    with gzip.open(filepath, 'rt', encoding='utf-8') as f:
        for line in f:
            if line.strip():
                records.append(json.loads(line))
    return records

# Example paths (adjust for Kaggle/Colab working directory):
train_path = 'data/splits/train.jsonl.gz'
dev_path = 'data/splits/dev.jsonl.gz'

if os.path.exists(train_path):
    train_records = load_jsonl_gz(train_path)
    dev_records = load_jsonl_gz(dev_path) if os.path.exists(dev_path) else []
    print(f'Loaded {len(train_records)} train records, {len(dev_records)} dev records.')
else:
    print(f'Split {train_path} not found. Ensure dataset splits are uploaded.')


## 3. Tokenization & Model Setup

Model backbone: `microsoft/deberta-v3-small` (44M parameters, fast inference, high representation quality).

In [ ]:
model_name = 'microsoft/deberta-v3-small'
tokenizer = AutoTokenizer.from_pretrained(model_name)

def preprocess_dataset(records, label2id):
    texts = [r['text'] for r in records]
    labels = [label2id[r['label']] if isinstance(r['label'], str) else r['label'] for r in records]
    encodings = tokenizer(texts, truncation=True, max_length=512, padding=False)
    encodings['labels'] = labels
    return Dataset.from_dict(encodings)

# 4-class taxonomy mapping
label2id = {'human': 0, 'human_refined': 1, 'ai_refined': 2, 'ai_generated': 3}
id2label = {v: k for k, v in label2id.items()}


## 4. Fine-Tuning Loop

In [ ]:
def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    preds = np.argmax(predictions, axis=1)
    acc = accuracy_score(labels, preds)
    macro_f1 = f1_score(labels, preds, average='macro')
    return {'accuracy': acc, 'macro_f1': macro_f1}

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=len(label2id),
    id2label=id2label,
    label2id=label2id
).to(device)

training_args = TrainingArguments(
    output_dir='./results_deberta_small',
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=3,
    weight_decay=0.01,
    evaluation_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='macro_f1',
    fp16=torch.cuda.is_available(),
    logging_steps=50,
)

# When ready to train:
# train_dataset = preprocess_dataset(train_records, label2id)
# dev_dataset = preprocess_dataset(dev_records, label2id)
# trainer = Trainer(
#     model=model,
#     args=training_args,
#     train_dataset=train_dataset,
#     eval_dataset=dev_dataset,
#     tokenizer=tokenizer,
#     data_collator=DataCollatorWithPadding(tokenizer=tokenizer),
#     compute_metrics=compute_metrics,
# )
# trainer.train()


## 5. Export to ONNX & Dynamic INT8 Quantization

In [ ]:
def export_and_quantize_onnx(model, tokenizer, output_dir='./exported_model'):
    os.makedirs(output_dir, exist_ok=True)
    onnx_fp32_path = os.path.join(output_dir, 'deberta_v3_small.onnx')
    onnx_int8_path = os.path.join(output_dir, 'deberta_v3_small_int8.onnx')

    model.eval()
    dummy_text = 'This is a sample sentence to initialize the dynamic ONNX computational graph.'
    dummy_inputs = tokenizer(dummy_text, return_tensors='pt').to(model.device)

    print('Exporting to FP32 ONNX...')
    torch.onnx.export(
        model,
        (dummy_inputs['input_ids'], dummy_inputs['attention_mask']),
        onnx_fp32_path,
        input_names=['input_ids', 'attention_mask'],
        output_names=['logits'],
        dynamic_axes={
            'input_ids': {0: 'batch_size', 1: 'sequence_length'},
            'attention_mask': {0: 'batch_size', 1: 'sequence_length'},
            'logits': {0: 'batch_size'}
        },
        opset_version=14,
        do_constant_folding=True
    )

    print('Quantizing to Dynamic INT8...')
    quantize_dynamic(
        model_input=onnx_fp32_path,
        model_output=onnx_int8_path,
        weight_type=QuantType.QInt8
    )

    fp32_size = os.path.getsize(onnx_fp32_path) / (1024 * 1024)
    int8_size = os.path.getsize(onnx_int8_path) / (1024 * 1024)
    print(f'FP32 Model Size: {fp32_size:.2f} MB')
    print(f'INT8 Model Size: {int8_size:.2f} MB')

# export_and_quantize_onnx(model, tokenizer)
